# 🚦 Road Accident Severity Prediction
## Data Mining & Data Warehousing (DMDW) Project

---

### Project Overview
This project implements an end-to-end machine learning pipeline for predicting the severity of road accidents (Fatal, Serious, or Slight) using the UK Road Accident dataset. It demonstrates key concepts from Data Mining and Data Warehousing:

- **Data Mining**: Classification using multiple ML algorithms, cross-validation, and evaluation
- **Data Warehousing**: Star schema design, ETL pipeline, and OLAP operations

### Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading & Generation](#2-data)
3. [Data Preprocessing](#3-preprocessing)
4. [Exploratory Data Analysis (EDA)](#4-eda)
5. [Model Training](#5-training)
6. [Model Evaluation](#6-evaluation)
7. [Star Schema & ETL Pipeline](#7-etl)
8. [Prediction Demo](#8-prediction)
9. [Conclusion](#9-conclusion)

### Tech Stack
- Python, Pandas, NumPy, Scikit-learn, XGBoost
- Matplotlib, Seaborn for visualizations
- Imbalanced-learn for SMOTE

---

## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib

# Add src to path for module imports
sys.path.insert(0, 'src')

print('✅ All libraries loaded successfully!')
print(f'   Pandas: {pd.__version__}')
print(f'   NumPy: {np.__version__}')

## 2. Data Loading & Generation <a id='2-data'></a>

We use a synthetic dataset modeled after the **UK Department for Transport (DfT) Road Accident dataset**.

> **Note**: For the real dataset, visit: https://www.kaggle.com/datasets/silicon99/dft-accident-data

### Key Features:
| Feature | Description |
|---|---|
| `Day_of_Week` | 1=Sunday to 7=Saturday |
| `Time` | Time of accident (HH:MM) |
| `Road_Type` | Single carriageway, Dual carriageway, etc. |
| `Speed_limit` | Speed limit in mph |
| `Weather_Conditions` | Fine, Raining, Snowing, etc. |
| `Road_Surface_Conditions` | Dry, Wet, Snow, etc. |
| `Light_Conditions` | Daylight, Darkness variations |
| `Number_of_Vehicles` | Vehicles involved |
| `Number_of_Casualties` | Casualties in the accident |
| `Urban_or_Rural_Area` | 1=Urban, 2=Rural |
| **`Accident_Severity`** | **Target: 1=Fatal, 2=Serious, 3=Slight** |

In [ ]:
# Load or generate the dataset
data_path = 'data/road_accidents.csv'

if not os.path.exists(data_path):
    print('Dataset not found. Generating synthetic dataset...')
    from data.generate_dataset import generate_dataset
    df = generate_dataset()
    df.to_csv(data_path, index=False)
else:
    df = pd.read_csv(data_path)
    print(f'✅ Dataset loaded: {df.shape[0]:,} records, {df.shape[1]} columns')

# Display first few rows
df.head()

In [ ]:
# Basic dataset info
print('📐 Shape:', df.shape)
print('\n📊 Data Types:')
print(df.dtypes)
print('\n📋 Statistical Summary:')
df.describe()

## 3. Data Preprocessing <a id='3-preprocessing'></a>

### Steps:
1. Handle missing values (mode/median imputation)
2. Feature engineering (extract hour, create flags)
3. Encode categorical variables (Label Encoding)
4. Normalize numerical features (StandardScaler)
5. Handle class imbalance (SMOTE)

In [ ]:
# Run the complete preprocessing pipeline
from data_preprocessing import preprocess_pipeline

preprocessed = preprocess_pipeline(data_path)

In [ ]:
# Examine preprocessed data
print(f'Training set shape: {preprocessed["X_train"].shape}')
print(f'Test set shape:     {preprocessed["X_test"].shape}')
print(f'\nFeatures used ({len(preprocessed["feature_names"])}): {preprocessed["feature_names"]}')

## 4. Exploratory Data Analysis (EDA) <a id='4-eda'></a>

Visualizing the dataset to understand patterns and relationships.

In [ ]:
# Use the raw dataframe for EDA (before encoding)
raw_df = pd.read_csv(data_path)

# Extract hour for analysis
raw_df['Hour'] = raw_df['Time'].apply(
    lambda x: int(str(x).split(':')[0]) if pd.notna(x) and ':' in str(x) else 12
)

SEVERITY_COLORS = {1: '#e74c3c', 2: '#f39c12', 3: '#2ecc71'}
SEVERITY_LABELS = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
PALETTE = ['#e74c3c', '#f39c12', '#2ecc71']

In [ ]:
# --- Plot 1: Severity Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Accident Severity Distribution', fontsize=16, fontweight='bold')

counts = raw_df['Accident_Severity'].value_counts().sort_index()
labels = [SEVERITY_LABELS[i] for i in counts.index]
colors = [SEVERITY_COLORS[i] for i in counts.index]

bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel('Number of Accidents', fontsize=12)
axes[0].set_title('Count by Severity')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion by Severity')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Correlation Heatmap ---
num_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
exclude = ['Latitude', 'Longitude', 'Year', 'Month']
num_cols = [c for c in num_cols if c not in exclude]

fig, ax = plt.subplots(figsize=(10, 8))
corr = raw_df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Accidents by Hour ---
fig, ax = plt.subplots(figsize=(12, 5))
for sev in sorted(raw_df['Accident_Severity'].unique()):
    subset = raw_df[raw_df['Accident_Severity'] == sev]
    hourly = subset.groupby('Hour').size()
    ax.plot(hourly.index, hourly.values, marker='o', linewidth=2,
            color=SEVERITY_COLORS[sev], label=SEVERITY_LABELS[sev], markersize=5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Accidents')
ax.set_title('Accidents by Hour of Day', fontsize=16, fontweight='bold')
ax.set_xticks(range(24))
ax.legend(title='Severity')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Weather Conditions ---
fig, ax = plt.subplots(figsize=(12, 6))
weather_sev = raw_df.groupby(['Weather_Conditions', 'Accident_Severity']).size().unstack(fill_value=0)
weather_sev.columns = [SEVERITY_LABELS.get(c, c) for c in weather_sev.columns]
weather_sev.sort_values(weather_sev.columns[-1], ascending=True).plot(
    kind='barh', stacked=True, ax=ax, color=PALETTE, edgecolor='white'
)
ax.set_xlabel('Number of Accidents')
ax.set_title('Accidents by Weather Conditions', fontsize=16, fontweight='bold')
ax.legend(title='Severity', loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 5: Speed Limit Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Speed Limit Analysis', fontsize=16, fontweight='bold')

sev_labels_list = [SEVERITY_LABELS[s] for s in raw_df['Accident_Severity']]
sns.boxplot(x=sev_labels_list, y=raw_df['Speed_limit'], ax=axes[0],
            palette=PALETTE, order=['Fatal', 'Serious', 'Slight'])
axes[0].set_title('Speed Limit by Severity')

speed_sev = raw_df.groupby(['Speed_limit', 'Accident_Severity']).size().unstack(fill_value=0)
speed_sev.columns = [SEVERITY_LABELS[c] for c in speed_sev.columns]
speed_sev.plot(kind='bar', ax=axes[1], color=PALETTE, edgecolor='white')
axes[1].set_title('Count by Speed Limit')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 6: Urban vs Rural ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Urban vs Rural Comparison', fontsize=16, fontweight='bold')

area_map = {1: 'Urban', 2: 'Rural'}
raw_df['Area'] = raw_df['Urban_or_Rural_Area'].map(area_map)

area_counts = raw_df['Area'].value_counts()
axes[0].pie(area_counts.values, labels=area_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#e67e22'], textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Accidents by Area')

area_sev = raw_df.groupby(['Area', 'Accident_Severity']).size().unstack(fill_value=0)
area_pct = area_sev.div(area_sev.sum(axis=1), axis=0) * 100
area_pct.columns = [SEVERITY_LABELS[c] for c in area_pct.columns]
area_pct.plot(kind='bar', ax=axes[1], color=PALETTE, edgecolor='white')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Severity % by Area')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 5. Model Training <a id='5-training'></a>

### Classification Algorithms:
1. **Decision Tree** — Simple, interpretable tree-based splits
2. **Random Forest** — Ensemble of decision trees (bagging)
3. **XGBoost** — Gradient boosting for state-of-the-art performance
4. **KNN** — Distance-based classification
5. **Naive Bayes** — Probabilistic classifier using Bayes' theorem

### Cross Validation:
We use **5-Fold Stratified Cross Validation** to ensure each fold maintains the class distribution.

In [ ]:
# Run the complete training pipeline
from model_training import run_training

training_results = run_training(preprocessed)

In [ ]:
# Display cross-validation results
print('📊 Cross-Validation Results (5-Fold):\n')
cv_df = training_results['cv_results']
cv_df.style.format({'Mean Accuracy': '{:.4f}', 'Std Accuracy': '{:.4f}',
                     'Min Accuracy': '{:.4f}', 'Max Accuracy': '{:.4f}'})

## 6. Model Evaluation <a id='6-evaluation'></a>

### Metrics:
- **Accuracy**: Overall correct predictions
- **Precision**: Of predicted positives, how many are correct
- **Recall**: Of actual positives, how many are found
- **F1-Score**: Harmonic mean of precision and recall
- **ROC-AUC**: Area under the ROC curve (multi-class, One-vs-Rest)

In [ ]:
# Run comprehensive evaluation
from evaluation import run_evaluation

comparison = run_evaluation(
    trained_models=training_results['trained_models'],
    X_test=preprocessed['X_test'],
    y_test=preprocessed['y_test'],
    feature_names=preprocessed['feature_names'],
    cv_results=training_results['cv_results']
)

In [ ]:
# Display final comparison table
comparison[['Model', 'Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1-Score (macro)']]

## 7. Star Schema & ETL Pipeline <a id='7-etl'></a>

### Data Warehouse Concepts

**Star Schema** — A data warehouse schema with:
- **Fact Table**: Contains measurable, quantitative data (accidents)
- **Dimension Tables**: Contains descriptive attributes (time, location, weather, road)

**ETL Pipeline**:
- **Extract**: Load raw data from source
- **Transform**: Clean, enrich, and reshape into star schema
- **Load**: Save to the data warehouse

```
┌─────────────┐     ┌──────────────┐
│  DIM_TIME    │     │ DIM_LOCATION  │
└──────┬──────┘     └──────┬───────┘
       │                   │
       ▼                   ▼
┌─────────────────────────────────┐
│       FACT_ACCIDENTS             │
│  (severity, vehicles,            │
│   casualties, speed_limit)       │
└──────┬──────────────┬───────────┘
       │              │
       ▼              ▼
┌──────────────┐ ┌──────────────┐
│ DIM_WEATHER   │ │  DIM_ROAD     │
└──────────────┘ └──────────────┘
```

In [ ]:
# Run the ETL pipeline
from star_schema_etl import run_etl_pipeline

warehouse_tables = run_etl_pipeline(data_path)

In [ ]:
# Inspect the dimension tables
print('📅 DIM_TIME (sample):')
display(warehouse_tables['dim_time'].head())

print('\n🌦️ DIM_WEATHER:')
display(warehouse_tables['dim_weather'])

print('\n🛣️ DIM_ROAD (sample):')
display(warehouse_tables['dim_road'].head())

print('\n📊 FACT_ACCIDENTS (sample):')
display(warehouse_tables['fact_accidents'].head())

## 8. Prediction Demo <a id='8-prediction'></a>

Using the trained model to predict accident severity for new inputs.

In [ ]:
from predict import predict_severity, load_model

# Load the saved best model
bundle = load_model()

# --- Scenario 1: Low-risk scenario ---
print('\n🟢 Scenario 1: Clear day, urban, low speed')
result1 = predict_severity(
    day_of_week=3, hour=10,
    road_type='Single carriageway', speed_limit=30,
    weather='Fine no high winds', road_surface='Dry',
    light='Daylight', num_vehicles=2, num_casualties=1,
    urban_rural=1, model_bundle=bundle
)
print(f'   Prediction: {result1["severity_label"]} — {result1["description"]}')
print(f'   Probabilities: {result1["probabilities"]}')

# --- Scenario 2: High-risk scenario ---
print('\n🔴 Scenario 2: Dark, rainy, rural, high speed')
result2 = predict_severity(
    day_of_week=7, hour=23,
    road_type='Single carriageway', speed_limit=60,
    weather='Raining no high winds', road_surface='Wet or damp',
    light='Darkness - no lighting', num_vehicles=1, num_casualties=3,
    urban_rural=2, model_bundle=bundle
)
print(f'   Prediction: {result2["severity_label"]} — {result2["description"]}')
print(f'   Probabilities: {result2["probabilities"]}')

## 9. Conclusion <a id='9-conclusion'></a>

### Key Findings:

1. **Class Imbalance**: ~84% of accidents are "Slight", making minority class prediction challenging
2. **Best Features**: Number of casualties, speed limit, and light conditions are the most predictive
3. **Model Performance**: Random Forest and XGBoost perform best overall
4. **Data Warehouse**: Star schema effectively organizes accident data for analytical queries

### Future Improvements:
- Use the real UK DfT dataset (1.7M+ records) for better model training
- Implement hyperparameter tuning (GridSearchCV / Optuna)
- Add geographic features (region, junction details)
- Deploy as a web application (Flask / Streamlit)
- Add real-time data integration for the ETL pipeline

---
*Project completed as part of DMDW coursework*